# baseline: color + edge features -> scikit-learn classifier (M3)

Hand-crafted features (HSV histogram, Canny edge density, local binary patterns) feed a
scikit-learn classifier. It predicts the coarse targets from `labels.ipynb`:
**occupancy** (empty/occupied) and **color3** (empty/white/black). It does not tell piece
types apart, so it is compared with the CNN at the occupancy / color level, not on all 13
classes.

Milestone: M3 (baseline vs CNN).

**TODO** (see `docs/M3_TODO.md`)
- [ ] `hsv_histogram`
- [ ] `edge_density` (Canny)
- [ ] `lbp_histogram` (skimage or numpy; add the dependency if skimage)
- [ ] turn the TODO notes in the Self-check into asserts once the features exist
- [ ] `train_baseline`: val evaluation per target, write `results/<run>/baseline.json`
- [ ] `make_estimator`: tune hyperparameters on the val sessions

In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np

%run ./labels.ipynb

In [ ]:
TARGETS = {"occupancy": ("y_occ", OCCUPANCY), "color3": ("y_color3", COLOR3)}

## Features
Each feature maps one RGB crop (H, W, 3) uint8 to a 1-D vector. Ideas: empty squares are
flat (few edges, one dominant color); white vs black pieces differ in brightness/hue.

In [ ]:
def hsv_histogram(crop: np.ndarray, bins: tuple[int, int, int] = (8, 4, 4)) -> np.ndarray:
    """Normalized 3D HSV histogram of an RGB crop, flattened (prod(bins),).

    Hint: cv2.cvtColor(..., COLOR_RGB2HSV), cv2.calcHist, divide by the pixel count.
    TODO M3: maybe only use the lower part of the crop (the square itself) for occupancy and
    the upper part (the piece body) for color.
    """
    raise NotImplementedError("TODO M3")


def edge_density(crop: np.ndarray, low: int = 50, high: int = 150, grid: int = 2) -> np.ndarray:
    """Fraction of Canny edge pixels, overall and per cell of a grid x grid split
    (1 + grid*grid,). Empty squares have few edges; pieces have many.
    """
    raise NotImplementedError("TODO M3")


def lbp_histogram(crop: np.ndarray, points: int = 8, radius: int = 1) -> np.ndarray:
    """Histogram of uniform local binary patterns on the gray crop (points + 2,).

    Either skimage.feature.local_binary_pattern (adds scikit-image dependency) or a small
    numpy implementation. TODO M3: pick one.
    """
    raise NotImplementedError("TODO M3")


FEATURES = {"hsv_hist": hsv_histogram, "edge_density": edge_density, "lbp": lbp_histogram}


def extract_features(crops, features: list[str]) -> np.ndarray:
    """Stack the named features for each crop -> (N, D) float32."""
    unknown = set(features) - set(FEATURES)
    if unknown:
        raise ValueError(f"unknown features {sorted(unknown)}; choose from {sorted(FEATURES)}")
    rows = [np.concatenate([np.ravel(FEATURES[f](c)) for f in features]) for c in crops]
    return np.asarray(rows, dtype=np.float32)

## Classifier
StandardScaler + one sklearn model per target, on shared features. Save/load uses pickle;
because the class is defined in a notebook, `%run ./baseline.ipynb` before loading a
pickle (pickle stores the class as `__main__.BaselineClassifier`).

In [ ]:
def make_estimator(kind: str):
    """StandardScaler + the chosen sklearn classifier, with predict_proba enabled."""
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVC

    # TODO M3: tune hyperparameters on the val sessions (C / gamma, n_estimators, ...).
    models = {
        # SVC(probability=True) is deprecated in recent sklearn; calibrate explicitly instead.
        "svm": lambda: CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced"), ensemble=False),
        "random_forest": lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced"),
        "logistic": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    }
    if kind not in models:
        raise ValueError(f"unknown classifier {kind!r}; choose from {sorted(models)}")
    return make_pipeline(StandardScaler(), models[kind]())


class BaselineClassifier:
    """One sklearn model per target ("occupancy", "color3") on shared features."""

    def __init__(self, features: list[str], classifier: str = "svm",
                 targets: tuple[str, ...] = ("occupancy", "color3")):
        for t in targets:
            if t not in TARGETS:
                raise ValueError(f"unknown target {t!r}; choose from {sorted(TARGETS)}")
        self.features = list(features)
        self.classifier = classifier
        self.targets = tuple(targets)
        self.models: dict = {}

    def featurize(self, crops) -> np.ndarray:
        return extract_features(crops, self.features)

    def fit(self, crops, y: dict[str, np.ndarray]) -> "BaselineClassifier":
        """y maps target name -> int labels (e.g. df["y_occ"]). Features are computed once."""
        X = self.featurize(crops)
        for t in self.targets:
            self.models[t] = make_estimator(self.classifier).fit(X, np.asarray(y[t]))
        return self

    def predict_proba(self, crops, target: str = "color3") -> np.ndarray:
        """(N, len(OCCUPANCY)) or (N, len(COLOR3)) probabilities for one target.

        Columns follow the class order in OCCUPANCY / COLOR3. Note: this is not the
        (N, 13) model contract; the baseline cannot see piece types.
        TODO M3 (optional): if a class is missing from training data, sklearn drops its column;
        re-insert zeros using self.models[target].classes_.
        """
        return self.models[target].predict_proba(self.featurize(crops))

    def predict(self, crops, target: str = "color3") -> np.ndarray:
        return self.models[target].predict(self.featurize(crops))

    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @staticmethod
    def load(path: str | Path) -> "BaselineClassifier":
        with open(path, "rb") as f:
            obj = pickle.load(f)
        if not isinstance(obj, BaselineClassifier):
            raise TypeError(f"{path} does not contain a BaselineClassifier")
        return obj

## Training entry point
Called from `run_train_baseline.ipynb`. Needs `dataset.ipynb` (`%run` it there; it is
not pulled in here to keep this notebook light).

In [ ]:
def train_baseline(cfg, out_path: str | Path = "models/baseline/baseline.pkl") -> BaselineClassifier:
    """Fit the baseline on cfg.data.train_sessions, report val accuracy, save to out_path.

    Requires read_crop / split_by_session from dataset.ipynb in the namespace.
    """
    splits = split_by_session(cfg)  # noqa: F821  (from dataset.ipynb)
    train = splits["train"]
    if len(train) == 0:
        raise ValueError("cfg.data.train_sessions is empty")
    b = cfg.baseline
    clf = BaselineClassifier(b.features, b.classifier, tuple(b.targets))
    crops = [read_crop(p) for p in train["abs_path"]]  # noqa: F821
    clf.fit(crops, {t: train[TARGETS[t][0]].to_numpy() for t in clf.targets})

    # TODO M3: evaluate on splits["val"] per target (accuracy, confusion matrix) and write
    # the numbers somewhere evaluate.ipynb can pick them up (results/<run_name>/baseline.json).
    clf.save(out_path)
    return clf

## Self-check
(was tests/test_baseline.py). A temporary `mean_rgb` feature stands in for the TODO
features so the classifier plumbing can be tested now.

TODO notes from skipped tests (turn into asserts when the features exist):
- `hsv_histogram`: shape == prod(bins), sums to 1; a pure-red crop puts all mass in one hue bin.
- `edge_density`: flat crop -> all zeros; checkerboard pattern -> high density; shape 1 + grid*grid.
- `lbp_histogram`: shape points + 2, sums to 1; constant image puts all mass in one bin.
- `train_baseline` end to end: tiny fake session with real crop PNGs -> writes the pickle and
  reports val accuracy.

In [ ]:
import tempfile


def _bl_mean_rgb(crop):
    return crop.reshape(-1, 3).mean(axis=0) / 255.0


def _bl_fake_data(n=60, seed=0):
    """Dark crops = empty, bright = white piece, red = black piece (stand-in for real data)."""
    rng = np.random.default_rng(seed)
    crops, color3 = [], []
    for i in range(n):
        c = i % 3
        base = {0: (20, 20, 20), 1: (230, 230, 230), 2: (200, 30, 30)}[c]
        img = np.clip(np.array(base) + rng.normal(0, 10, (20, 10, 3)), 0, 255).astype(np.uint8)
        crops.append(img)
        color3.append(c)
    color3 = np.array(color3)
    return crops, {"color3": color3, "occupancy": (color3 > 0).astype(int)}


def _bl_raises(exc, fn, *a, **k):
    try:
        fn(*a, **k)
    except exc:
        return
    raise AssertionError(f"{fn} did not raise {exc.__name__}")


# unknown feature / target / classifier
_bl_raises(ValueError, extract_features, [np.zeros((4, 4, 3), np.uint8)], ["nope"])
_bl_raises(ValueError, BaselineClassifier, ["hsv_hist"], targets=("pieces",))
_bl_raises(ValueError, make_estimator, "xgboost")

# fit / predict / save / load for each classifier kind
FEATURES["mean_rgb"] = _bl_mean_rgb
try:
    _crops, _y = _bl_fake_data()
    for _kind in ["svm", "random_forest", "logistic"]:
        _clf = BaselineClassifier(["mean_rgb"], classifier=_kind).fit(_crops, _y)
        _p = _clf.predict_proba(_crops[:6], target="color3")
        assert _p.shape == (6, 3) and np.allclose(_p.sum(axis=1), 1)
        assert _clf.predict_proba(_crops[:6], target="occupancy").shape == (6, 2)
        assert (_clf.predict(_crops, target="color3") == _y["color3"]).mean() > 0.9
        with tempfile.TemporaryDirectory() as _t:
            _clf.save(Path(_t) / "b.pkl")
            _clf2 = BaselineClassifier.load(Path(_t) / "b.pkl")
        assert np.array_equal(_clf2.predict(_crops, "occupancy"), _clf.predict(_crops, "occupancy"))
finally:
    del FEATURES["mean_rgb"]
print("baseline OK")